In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:
# Load dataset
df = pd.read_csv("training.csv")

print(df.shape)
df.head()

(95662, 16)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [3]:
# Drop duplicates
df = df.drop_duplicates()

# Handle missing values
df = df.dropna(subset=["TransactionStartTime", "Amount", "Value"])

df.reset_index(drop=True, inplace=True)


In [4]:
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])

df["hour"] = df["TransactionStartTime"].dt.hour
df["dayofweek"] = df["TransactionStartTime"].dt.dayofweek


In [5]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "ProviderId", "ProductCategory", "ChannelId"
]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


In [6]:
from sklearn.preprocessing import RobustScaler

numerical_cols = ["Amount", "Value", "PricingStrategy", "hour", "dayofweek"]

scaler = RobustScaler()

df[numerical_cols] = df[numerical_cols].astype(float)
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])


In [7]:
feature_cols = categorical_cols + numerical_cols
X = df[feature_cols].values


In [8]:
SEQ_LEN = 10

def create_sequences(data, seq_len):
    sequences = []
    for i in range(len(data) - seq_len):
        sequences.append(data[i:i+seq_len])
    return np.array(sequences)

X_seq = create_sequences(X, SEQ_LEN)

print("Sequence shape:", X_seq.shape)


Sequence shape: (95652, 10, 8)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val = train_test_split(
    X_seq,
    test_size=0.2,
    shuffle=False
)

print(X_train.shape, X_val.shape)


(76521, 10, 8) (19131, 10, 8)


In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn_lstm_autoencoder(T, F):
    inp = layers.Input(shape=(T, F))

    x = layers.Conv1D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling1D(2)(x)

    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.LSTM(32, return_sequences=True)(x)

    x = layers.UpSampling1D(2)(x)
    out = layers.TimeDistributed(layers.Dense(F))(x)

    model = models.Model(inp, out)
    model.compile(optimizer="adam", loss="mse")

    return model


In [11]:
def build_cnn_gru_autoencoder(T, F):
    inputs = layers.Input(shape=(T, F))

    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    encoded = layers.GRU(64, return_sequences=True)(x)
    decoded = layers.GRU(64, return_sequences=True)(encoded)

    outputs = layers.TimeDistributed(layers.Dense(F))(decoded)

    model = models.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")

    return model


In [12]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

cnn_lstm = build_cnn_lstm_autoencoder(SEQ_LEN, X_seq.shape[2])

history_lstm = cnn_lstm.fit(
    X_train, X_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)


Epoch 1/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 134.6968 - val_loss: 1018.4047
Epoch 2/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 15s 12ms/step - loss: 127.3578 - val_loss: 995.8802
Epoch 3/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - loss: 122.2818 - val_loss: 976.9554
Epoch 4/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 118.2278 - val_loss: 960.7471
Epoch 5/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - loss: 114.7833 - val_loss: 945.1577
Epoch 6/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 111.7300 - val_loss: 931.0216
Epoch 7/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 109.0446 - val_loss: 917.7234
Epoch 8/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 106.6206 - val_loss: 905.1077
Epoch 9/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 104.3871 - val_loss: 893.7614
Epoch 10/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 102.4523 - val_loss: 882.4796
Epoch 11/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step -

In [13]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

cnn_gru  = build_cnn_gru_autoencoder(SEQ_LEN, X_train.shape[2])

history_gru = cnn_gru.fit(
    X_train, X_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, X_val),
    callbacks=[early_stop]
)


Epoch 1/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 32s 21ms/step - loss: 126.8939 - val_loss: 980.2452
Epoch 2/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 23s 19ms/step - loss: 112.5634 - val_loss: 934.7302
Epoch 3/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - loss: 102.5732 - val_loss: 896.2776
Epoch 4/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - loss: 94.6206 - val_loss: 862.3030
Epoch 5/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - loss: 87.7397 - val_loss: 831.1568
Epoch 6/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 21ms/step - loss: 82.1585 - val_loss: 804.0930
Epoch 7/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - loss: 76.8768 - val_loss: 777.3253
Epoch 8/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 21ms/step - loss: 72.3706 - val_loss: 753.1098
Epoch 9/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 21ms/step - loss: 68.5745 - val_loss: 731.1893
Epoch 10/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - loss: 64.5962 - val_loss: 710.3293
Epoch 11/50
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - loss: 6

In [14]:

# Predict reconstruction on validation set
X_val_pred_lstm = cnn_lstm.predict(X_val)
X_val_pred_gru  = cnn_gru.predict(X_val)

# Compute reconstruction errors
recon_err_lstm = np.mean(np.square(X_val - X_val_pred_lstm), axis=(1, 2))
recon_err_gru  = np.mean(np.square(X_val - X_val_pred_gru), axis=(1, 2))

# Define thresholds (99th percentile – standard choice)
lstm_threshold = np.percentile(recon_err_lstm, 99)
gru_threshold  = np.percentile(recon_err_gru, 99)

print("LSTM Threshold:", lstm_threshold)
print("GRU Threshold:", gru_threshold)



598/598 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
598/598 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step
LSTM Threshold: 6407.097989196521
GRU Threshold: 228.5625404485626


In [17]:
X_val_pred = cnn_lstm.predict(X_val)

recon_err = np.mean(
    np.square(X_val - X_val_pred),
    axis=(1,2)
)




598/598 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [18]:
threshold = np.percentile(recon_err, 95)
print("Initial Threshold:", threshold)


Initial Threshold: 43.84349466684591


In [19]:
y_pred = (recon_err > threshold).astype(int)

print("Anomalies detected:", np.sum(y_pred))


Anomalies detected: 957


In [20]:
feedback_idx = np.where(y_pred == 1)[0]

confirmed_labels = np.zeros_like(recon_err)
false_positive_idx = np.random.choice(
    feedback_idx,
    size=int(0.7 * len(feedback_idx)),
    replace=False
)

confirmed_labels[false_positive_idx] = 0
confirmed_labels[np.setdiff1d(feedback_idx, false_positive_idx)] = 1


In [21]:
confirmed_normal_errors = recon_err[confirmed_labels == 0]

adaptive_threshold = np.percentile(confirmed_normal_errors, 97)
print("Adaptive Threshold:", adaptive_threshold)


Adaptive Threshold: 65.90177756243375


In [22]:
y_pred_updated = (recon_err > adaptive_threshold).astype(int)

print("Original anomalies:", np.sum(y_pred))
print("Updated anomalies:", np.sum(y_pred_updated))


Original anomalies: 957
Updated anomalies: 812
